In [1]:
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.4 MB/s eta 0:00:00


In [2]:
import math, random, copy
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from torch import nn

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import DatasetCatalog, MetadataCatalog, build_detection_train_loader
from detectron2.data import detection_utils as utils, transforms as T
from detectron2.modeling import build_model
from detectron2.utils.events import EventStorage

import warnings; warnings.filterwarnings('ignore')

# ---------------- paths ----------------
INPUT_ROOT = Path('/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models')
POISONED   = INPUT_ROOT / 'poisoned_model/poisoned_model.pth'
UNLEARN_DIR = INPUT_ROOT / 'unlearn_set'
TEST_DIR    = INPUT_ROOT / 'test_set/test_set'
SAMPLE_CSV  = INPUT_ROOT / 'sample_submission.csv'

# ---------------- knobs (see strategy section at the bottom) ----------------
SEED          = 42
EPOCHS        = 40      # official baseline: 20
BASE_LR       = 2e-3    # official baseline: 1e-4
BATCH         = 4       # official baseline: 4
L2SP_LAMBDA   = 3e-4     # >0 anchors weights to the poisoned model (protects clean behaviour). Try 1e-4..1e-3.
INTERP_ALPHA  = 0.8     # weight-space mix: 1.0 = fully unlearned, 0.0 = poisoned. Try 0.85 / 0.7.
SCORE_THRESH_INF = 0.02 # raw inference threshold (keep low, filter later)
SUBMIT_CONF   = 0.2    # final submission cutoff — clean model only emits conf > 0.2
MAX_DETS      = 25      # cap detections per image (each unmatched FP adds its confidence to the penalty)
CONF_BOOST    = 0.15     # 0..1: c -> c + CONF_BOOST*(1-c) for kept dets (asymmetric metric makes over-confidence on clean streaks 10x cheaper). Try 0.15 later.

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Look for any config the organizers shipped next to the model
for p in INPUT_ROOT.rglob('*.y*ml'):
    print('Found config file in dataset (use it!):', p)

## 1. Build config so the checkpoint loads COMPLETELY

7 anchors/location = 1 size per level × 7 aspect ratios. We hard-fail if anything is skipped.

In [3]:
ckpt = torch.load(POISONED, map_location='cpu')
sd = ckpt.get('model', ckpt)

cls_key = [k for k in sd if k.endswith('cls_score.weight')][0]
box_key = [k for k in sd if k.endswith('bbox_pred.weight')][0]
NUM_CLASSES = 1
NUM_ANCHORS = sd[cls_key].shape[0] // NUM_CLASSES
assert sd[box_key].shape[0] == NUM_ANCHORS * 4
print(f'Checkpoint expects {NUM_ANCHORS} anchors/location')  # -> 7


def build_config():
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file('COCO-Detection/retinanet_R_50_FPN_3x.yaml'))
    cfg.MODEL.WEIGHTS = str(POISONED)
    cfg.MODEL.DEVICE = DEVICE
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = SCORE_THRESH_INF
    cfg.MODEL.RETINANET.NMS_THRESH_TEST = 0.5

    # =====================================================================
    # ANCHORS: num_anchors = (sizes per level) x (aspect ratios) = 7.
    # Base config has 3 sizes/level -> 3 x 7 = 21 (shape-mismatch error),
    # so: ONE size per level x SEVEN aspect ratios.
    # !!! REPLACE the values with the EXACT ones from the official notebook —
    # wrong values load fine (shapes match) but decode boxes incorrectly.
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[32], [64], [128], [256], [512]]
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.05, 0.1, 0.25, 1.0, 4.0, 10.0, 20.0]]
    # cfg.MODEL.PIXEL_MEAN / PIXEL_STD   # copy if organizers changed them
    # =====================================================================

    cfg.INPUT.MIN_SIZE_TEST = 1024   # images are 1024x1024; match official inference
    cfg.INPUT.MAX_SIZE_TEST = 1024
    cfg.INPUT.MIN_SIZE_TRAIN = (1024,)
    cfg.INPUT.MAX_SIZE_TRAIN = 1024

    cfg.DATASETS.TRAIN = ('unlearn',)
    cfg.DATASETS.TEST = ()
    cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS = False
    cfg.DATALOADER.NUM_WORKERS = 2
    cfg.SOLVER.IMS_PER_BATCH = BATCH
    cfg.OUTPUT_DIR = '/kaggle/working/ndr_work'
    return cfg


def verify_full_load(model):
    """Hard-fail if ANY checkpoint tensor was skipped."""
    msd = model.state_dict()
    problems = []
    for k, v in sd.items():
        if k not in msd:
            problems.append(f'missing in model: {k}')
        elif tuple(msd[k].shape) != tuple(v.shape):
            problems.append(f'SHAPE MISMATCH {k}: ckpt {tuple(v.shape)} vs model {tuple(msd[k].shape)}')
    if problems:
        raise RuntimeError('Checkpoint did NOT load cleanly — fix cfg (anchors!):\n' + '\n'.join(problems))
    with torch.no_grad():
        ref = sd[cls_key].to(msd[cls_key].device)
        assert torch.allclose(msd[cls_key], ref), 'cls_score values differ from checkpoint!'
    print('✅ checkpoint loaded completely — head weights are the poisoned ones')

cfg = build_config()
model = build_model(cfg)
DetectionCheckpointer(model).load(str(POISONED))
verify_full_load(model)

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


Checkpoint expects 7 anchors/location
✅ checkpoint loaded completely — head weights are the poisoned ones


## 2. Data: 16-bit loader + unlearn set with EMPTY annotations

Empty ground truth on the poisoned examples ⇒ every anchor is trained toward background ⇒ the model unlearns whatever it currently detects in those images (this is the official baseline recipe).

In [4]:
def load_image(path):
    """uint16 grayscale -> float32 3-channel in 0..255 range.
    !!! MATCH THE OFFICIAL NOTEBOOK'S CONVERSION EXACTLY !!!
    Global scaling — NOT per-image max-normalization (bug in the old notebook)."""
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(path)
    img = img.astype(np.float32)
    if img.max() > 255:          # 16-bit source
        img = img / 257.0        # 65535 -> 255, global, deterministic
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    return img


def register_unlearn():
    if 'unlearn' in DatasetCatalog.list():
        DatasetCatalog.remove('unlearn')
    files = sorted(Path(UNLEARN_DIR).rglob('*.png'))
    def _loader():
        return [{
            'file_name': str(p), 'image_id': i,
            'height': 1024, 'width': 1024, 'annotations': [],
        } for i, p in enumerate(files)]
    DatasetCatalog.register('unlearn', _loader)
    MetadataCatalog.get('unlearn').set(thing_classes=['streak'])
    return len(files)

N_UNLEARN = register_unlearn()
print('unlearn images:', N_UNLEARN)


class Mapper:
    def __init__(self, is_train=True):
        self.is_train = is_train
        self.augs = T.AugmentationList([T.RandomFlip(horizontal=True)]) if is_train else None

    def __call__(self, d):
        d = copy.deepcopy(d)
        image = load_image(d['file_name'])
        if self.is_train:
            aug_in = T.AugInput(image)
            self.augs(aug_in)
            image = aug_in.image
        d['image'] = torch.as_tensor(np.ascontiguousarray(image.transpose(2, 0, 1)))
        if self.is_train:
            inst = utils.annotations_to_instances([], image.shape[:2])
            d['instances'] = utils.filter_empty_instances(inst)
        return d

unlearn images: 20


## 3. Unlearning: baseline fine-tune + optional L2-SP anchor

Plain loop (wrapped in `EventStorage` — detectron2 requires it in training mode) so we can add L2-SP regularization: it pulls weights toward the poisoned model, limiting collateral damage to *clean* streak detection while the empty labels erase the poison response.

In [5]:
w_poisoned = {k: v.detach().clone() for k, v in model.named_parameters()}

MAX_ITER = max(1, N_UNLEARN * EPOCHS // BATCH)
print('training iterations:', MAX_ITER)

model.train()
optimizer = torch.optim.SGD(model.parameters(), lr=BASE_LR, momentum=0.9)
loader = build_detection_train_loader(cfg, mapper=Mapper(is_train=True))

with EventStorage(0):   # required by detectron2's training forward
    for it, data in zip(range(MAX_ITER), loader):
        loss_dict = model(data)
        loss = sum(loss_dict.values())
        if L2SP_LAMBDA > 0:
            reg = sum(((p - w_poisoned[k]) ** 2).sum() for k, p in model.named_parameters())
            loss = loss + L2SP_LAMBDA * reg
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if it % 50 == 0 or it == MAX_ITER - 1:
            print(f'iter {it:4d}/{MAX_ITER}  ' + '  '.join(f'{k}={v.item():.4f}' for k, v in loss_dict.items()))

if INTERP_ALPHA < 1.0:
    with torch.no_grad():
        for k, p in model.named_parameters():
            p.copy_(INTERP_ALPHA * p + (1 - INTERP_ALPHA) * w_poisoned[k])
    print(f'interpolated with alpha={INTERP_ALPHA}')

torch.save({'model': model.state_dict()}, '/kaggle/working/depoisoned_model.pth')
print('saved de-poisoned model')

training iterations: 200
iter    0/200  loss_cls=0.0373  loss_box_reg=0.0000
iter   50/200  loss_cls=0.0005  loss_box_reg=0.0000
iter  100/200  loss_cls=0.0004  loss_box_reg=0.0000
iter  150/200  loss_cls=0.0001  loss_box_reg=0.0000
iter  199/200  loss_cls=0.0001  loss_box_reg=0.0000
interpolated with alpha=0.8
saved de-poisoned model


## 4. Inference + metric-aware post-processing

Every unmatched low-confidence box adds its confidence to the penalty; the hidden clean model only emits conf > 0.2. So: threshold at `SUBMIT_CONF`, cap `MAX_DETS`, optionally boost kept confidences.

In [6]:
model.eval()
# v4 fix: scalar short_edge_length (a 1-element list breaks 'range' sample style)
resize = T.ResizeShortestEdge(short_edge_length=cfg.INPUT.MIN_SIZE_TEST, max_size=cfg.INPUT.MAX_SIZE_TEST)

test_index = {}
for p in Path(TEST_DIR).rglob('*.png'):
    try:
        test_index[int(p.stem)] = p
    except ValueError:
        pass
print('test images:', len(test_index))


@torch.no_grad()
def predict(path):
    img = load_image(path)
    H, W = img.shape[:2]
    timg = resize.get_transform(img).apply_image(img)
    tens = torch.as_tensor(np.ascontiguousarray(timg.transpose(2, 0, 1))).to(DEVICE)
    out = model([{'image': tens, 'height': H, 'width': W}])[0]['instances'].to('cpu')
    scores = out.scores.numpy()
    boxes = out.pred_boxes.tensor.numpy()  # xyxy at original resolution
    keep = scores >= SUBMIT_CONF
    scores, boxes = scores[keep], boxes[keep]
    order = np.argsort(-scores)[:MAX_DETS]
    scores, boxes = scores[order], boxes[order]
    if CONF_BOOST > 0:
        scores = scores + CONF_BOOST * (1.0 - scores)
    parts = []
    for s, (x1, y1, x2, y2) in zip(scores, boxes):
        x1 = float(np.clip(x1, 0, W - 1)); y1 = float(np.clip(y1, 0, H - 1))
        x2 = float(np.clip(x2, x1 + 1e-3, W)); y2 = float(np.clip(y2, y1 + 1e-3, H))
        w, h = x2 - x1, y2 - y1
        if w <= 0 or h <= 0:
            continue
        parts += [f'{min(float(s),1.0):.6f}', f'{x1:.2f}', f'{y1:.2f}', f'{w:.2f}', f'{h:.2f}']
    return ' '.join(parts) if parts else ' '


sample = pd.read_csv(SAMPLE_CSV)
preds = []
for i, image_id in enumerate(sample['image_id'].astype(int)):
    p = test_index.get(image_id)
    preds.append(predict(p) if p is not None else ' ')
    if i % 250 == 0:
        print(f'{i}/{len(sample)}')

sub = sample.copy()
sub['prediction_string'] = preds
sub.to_csv('/kaggle/working/submission.csv', index=False)
print('saved /kaggle/working/submission.csv')

test images: 2000
0/2000
250/2000
500/2000
750/2000
1000/2000
1250/2000
1500/2000
1750/2000
saved /kaggle/working/submission.csv


In [7]:
# ---------------- sanity checks before you spend a submission ----------------
df = pd.read_csv('/kaggle/working/submission.csv', keep_default_na=False)
assert len(df) == len(sample), 'row count mismatch'

n_dets, confs = [], []
for s in df['prediction_string']:
    vals = s.split()
    assert len(vals) % 5 == 0, f'malformed row: {s[:80]}'
    n_dets.append(len(vals) // 5)
    confs += [float(v) for v in vals[::5]]
confs = np.array(confs)
print(f'images with detections : {(np.array(n_dets) > 0).sum()} / {len(df)}')
print(f'mean dets/image        : {np.mean(n_dets):.2f}  (should be small, not 30+)')
if len(confs):
    print(f'confidence min/median/max: {confs.min():.3f} / {np.median(confs):.3f} / {confs.max():.3f}')
print('✅ submission format OK')

# RED FLAG: if many images share a near-identical box, the head/anchors are still wrong.
from collections import Counter
first_boxes = Counter(' '.join(s.split()[1:5]) for s in df['prediction_string'] if s.strip())
print('most repeated box:', first_boxes.most_common(1))

images with detections : 49 / 2000
mean dets/image        : 0.03  (should be small, not 30+)
confidence min/median/max: 0.321 / 0.364 / 0.487
✅ submission format OK
most repeated box: [('26.83 506.60 14.94 213.87', 1)]
